In [1]:
# Importing libraries

import os
import zipfile
import pandas as pd
import whisper
import spacy

In [2]:
# ------------------------------------------------------------------------------
# STEP 1: INITIALIZE ENVIRONMENT AND CONSTANTS
# ------------------------------------------------------------------------------
# We establish our pathing variables. Using 'r' handles Windows backslashes seamlessly.
ZIP_FILE_PATH = r"C:\Users\imahaja2\Desktop\PhD_1(2024)\Year 2\GEN AI\Group Assignments\Assignment1_CrimeReport\audio_inputdataset_kaggle.zip"
EXTRACTION_FOLDER = "extracted_audio"
AUDIO_SUBFOLDER = os.path.join(EXTRACTION_FOLDER, "911_first6sec")
OUTPUT_CSV_NAME = "audio_extracted_report.csv"

In [3]:
# ------------------------------------------------------------------------------
# STEP 2: DATA EXTRACTION (UNZIPPING WORKSPACE)
# ------------------------------------------------------------------------------
print("--- [STAGE 1/5: DATA INGESTION] ---")
if os.path.exists(ZIP_FILE_PATH):
    # Open the archive and extract all raw audio content to our target directory
    with zipfile.ZipFile(ZIP_FILE_PATH, 'r') as zip_ref:
        zip_ref.extractall(EXTRACTION_FOLDER)
    print(f" Successfully unzipped archive to: '{EXTRACTION_FOLDER}'")
else:
    print(f" Error: Could not locate the zip archive at {ZIP_FILE_PATH}")
    # We raise an error to stop execution if the file path is incorrect
    raise FileNotFoundError("Verify your zip archive location before proceeding.")

# Verify and count our target file directory
if os.path.exists(AUDIO_SUBFOLDER):
    audio_files = [f for f in os.listdir(AUDIO_SUBFOLDER) if f.endswith('.wav')]
    print(f" Total target waveforms found for processing: {len(audio_files)}")
else:
    print(f" Error: Cannot find subfolder '{AUDIO_SUBFOLDER}' inside the extracted directory.")
    raise FileNotFoundError("Directory architecture mismatch.")

--- [STAGE 1/5: DATA INGESTION] ---
 Successfully unzipped archive to: 'extracted_audio'
 Total target waveforms found for processing: 707


In [4]:
# ------------------------------------------------------------------------------
# STEP 3: INITIALIZE DEEP LEARNING & NLP ENGINE WEIGHTS
# ------------------------------------------------------------------------------
print("\n--- [STAGE 2/5: INITIALIZING AI ARCHITECTURE] ---")

print(" Loading OpenAI Whisper Model ('base' configuration)...")
# 'base' strikes an optimal balance between execution speed and lexical accuracy on local CPUs
audio_model = whisper.load_model("base")

print(" Loading spaCy Language Processor Model ('en_core_web_sm')...")
# Small English pipeline optimized for fast Named Entity Recognition (NER) inference tasks
nlp_processor = spacy.load("en_core_web_sm")

print(" AI engines loaded successfully. Commencing ingestion loop.")


--- [STAGE 2/5: INITIALIZING AI ARCHITECTURE] ---
 Loading OpenAI Whisper Model ('base' configuration)...
 Loading spaCy Language Processor Model ('en_core_web_sm')...
 AI engines loaded successfully. Commencing ingestion loop.


In [5]:
# ------------------------------------------------------------------------------
# STEP 4: PROCESSING PIPELINE LOOP (TRANSCRIPTION, NER, AND FEATURES)
# ------------------------------------------------------------------------------
print("\n--- [STAGE 3/5: PROCESSING INTERMEDIATE FEATURES] ---")

# Memory buffer array to safely store extracted dictionary rows before converting to a DataFrame
all_records = []

for index, filename in enumerate(audio_files):
    # Construct absolute file pathing for each individual waveform
    current_file_path = os.path.join(AUDIO_SUBFOLDER, filename)
    
    # Batch status tracking: Prints progress tracking parameters every 10 updates to manage performance logs
    if (index + 1) % 10 == 0 or (index + 1) == len(audio_files):
        print(f" Progress Status: Completed processing ({index + 1} / {len(audio_files)}) files...")
        
    # --- TASK 4.1: AUTOMATIC SPEECH RECOGNITION (ASR) ---
    # Transcribe audio file binary array data into localized string formats using Whisper
    transcription_payload = audio_model.transcribe(current_file_path)
    raw_transcript = transcription_payload["text"].strip()
    
    # Lowercase normalization for standardized lookup evaluations
    normalized_text = raw_transcript.lower()
    
    # --- TASK 4.2: SAFETY FILTER & EXCEPTION HANDLING ---
    # Accounts for silent calls, pocket dial tones, or corrupted signals in raw streams
    if not raw_transcript or len(raw_transcript) < 3:
        final_transcript = "[Background noise / Dispatched tone / No clear speech]"
        detected_event = "Emergency Report"
        geographical_location = "Unknown Location"
        emotional_sentiment = "Calm"
        urgency_score = 0.20
    else:
        final_transcript = raw_transcript
        
        # --- TASK 4.3: HEURISTIC INTENT CLASSIFICATION ---
        # Evaluate contextual tokens against explicit category lexicons
        if any(keyword in normalized_text for keyword in ["fire", "smoke", "burning", "nelson building"]):
            detected_event = "Fire/Structure Incident"
        elif any(keyword in normalized_text for keyword in ["bank", "blocopia", "rob", "theft", "steal", "gun"]):
            detected_event = "Crime/Assault"
        elif any(keyword in normalized_text for keyword in ["accident", "crash", "car", "highway"]):
            detected_event = "Traffic Accident"
        else:
            detected_event = "Emergency Report"
            
        # --- TASK 4.4: NAMED ENTITY RECOGNITION (NER) VIA SPACY ---
        # Process the transcription text through our language dependency parser graph
        nlp_doc = nlp_processor(final_transcript)
        geographical_location = "Unknown Location"
        
        # Extract location entities (GPE=Geopolitical entities, FAC=Buildings/Facilities, ORG=Companies)
        for entity in nlp_doc.ents:
            if entity.label_ in ["GPE", "LOC", "FAC", "ORG"]:
                geographical_location = entity.text.title()
                break  # Retain the primary/first detected location token
                
        # Fallback overriding heuristic adjustments tailored specifically to this raw data domain
        if "blocopia bank" in normalized_text:
            geographical_location = "Blocopia Bank"
        elif "nelson building" in normalized_text:
            geographical_location = "Nelson Building"

        # --- TASK 4.5: SENTIMENT ANALYSIS & URGENCIES FEATURE ENGINEERING ---
        # Map qualitative situational assessments to deterministic mathematical parameters
        if detected_event in ["Fire/Structure Incident", "Crime/Assault"]:
            emotional_sentiment = "Distressed"
            urgency_score = 0.85  # Standardized high priority indicator score
        else:
            emotional_sentiment = "Calm"
            urgency_score = 0.40  # Standardized standard operational priority score

    # --- TASK 4.6: PRIMARY KEY SCHEMATIC FORMALIZATION ---
    # Establish unique identifiers mapping precisely to Student 6's orchestration module rules
    primary_key_id = f"AUD-{str(index + 1).zfill(3)}"  # Generates: AUD-001, AUD-002, etc.
    caller_id = f"C{str(index + 1).zfill(3)}"          # Generates: C001, C002, etc.
    
    # Store clean row record dictionary inside our collection array buffer
    all_records.append({
        "Incident_ID": primary_key_id,
        "Call_ID": caller_id,
        "Transcript": final_transcript,
        "Extracted_Event": detected_event,
        "Location": geographical_location,
        "Sentiment": emotional_sentiment,
        "Urgency_Score": urgency_score
    })


--- [STAGE 3/5: PROCESSING INTERMEDIATE FEATURES] ---


C:\Users\imahaja2\AppData\Local\anaconda3\Lib\site-packages\whisper\transcribe.py:132: UserWarning: FP16 is not supported on CPU; using FP32 instead
  warnings.warn("FP16 is not supported on CPU; using FP32 instead")


 Progress Status: Completed processing (10 / 707) files...
 Progress Status: Completed processing (20 / 707) files...
 Progress Status: Completed processing (30 / 707) files...
 Progress Status: Completed processing (40 / 707) files...
 Progress Status: Completed processing (50 / 707) files...
 Progress Status: Completed processing (60 / 707) files...
 Progress Status: Completed processing (70 / 707) files...
 Progress Status: Completed processing (80 / 707) files...
 Progress Status: Completed processing (90 / 707) files...
 Progress Status: Completed processing (100 / 707) files...
 Progress Status: Completed processing (110 / 707) files...
 Progress Status: Completed processing (120 / 707) files...
 Progress Status: Completed processing (130 / 707) files...
 Progress Status: Completed processing (140 / 707) files...
 Progress Status: Completed processing (150 / 707) files...
 Progress Status: Completed processing (160 / 707) files...
 Progress Status: Completed processing (170 / 707

In [6]:
# ------------------------------------------------------------------------------
# STEP 5: DATA SERIALIZATION & RENDERING
# ------------------------------------------------------------------------------
print("\n--- [STAGE 4/5: SERIALIZING STRUCTURED REPORT] ---")

# Convert unified collection dictionaries directly into a relational Pandas table layout
final_dataframe = pd.DataFrame(all_records)

# Export data matrix into disk space as a clean, standardized comma-separated values file
final_dataframe.to_csv(OUTPUT_CSV_NAME, index=False)
print(f" PIPELINE EXECUTION SUCCESSFUL! File saved as: '{OUTPUT_CSV_NAME}'")

print("\n--- [STAGE 5/5: PREVIEWING EXTRACTED DATA FRAME] ---")
# Render data visualization matrix directly onto your Jupyter screen viewport
final_dataframe


--- [STAGE 4/5: SERIALIZING STRUCTURED REPORT] ---
 PIPELINE EXECUTION SUCCESSFUL! File saved as: 'audio_extracted_report.csv'

--- [STAGE 5/5: PREVIEWING EXTRACTED DATA FRAME] ---


,Incident_ID,Call_ID,Transcript,Extracted_Event,Location,Sentiment,Urgency_Score
0,AUD-001,C001,one,Emergency Report,Unknown Location,Calm,0.40
1,AUD-002,C002,This is the Blocopia Bank in 1302 with the Tra...,Crime/Assault,Blocopia Bank,Distressed,0.85
2,AUD-003,C003,Yeah? Yeah. Yeah. Yeah. Yeah. Yeah.,Emergency Report,Unknown Location,Calm,0.40
3,AUD-004,C004,"Yes, we're at the Nelson building and there's ...",Fire/Structure Incident,Nelson Building,Distressed,0.85
4,AUD-005,C005,"Listen, we're playing really bad at this. Okay...",Emergency Report,Unknown Location,Calm,0.40
...,...,...,...,...,...,...,...
702,AUD-703,C703,Not that it's bailing the floor against Live P...,Emergency Report,Live Pls,Calm,0.40
703,AUD-704,C704,"Oh, my mind's at the doctor. You're not at the...",Emergency Report,Unknown Location,Calm,0.40
704,AUD-705,C705,"I spy here, John websites didn't show right up...",Emergency Report,Unknown Location,Calm,0.40
705,AUD-706,C706,"Oh, there's a student. Who's been shot, man?",Emergency Report,Unknown Location,Calm,0.40
